In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

project_root = '/content/drive/MyDrive/absa-multitask-roberta'
os.chdir(project_root)
sys.path.insert(0, project_root)

print("Current directory:", os.getcwd())

In [ ]:
!pip install transformers seqeval -q
print("Libraries installed.")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import json
import numpy as np
from tqdm import tqdm
from seqeval.metrics import f1_score
from sklearn.metrics import accuracy_score
import pandas as pd
import xml.etree.ElementTree as ET
import random
from utils.helpers import set_seed
from src.model import MultiTaskRoBERTa

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

In [ ]:
# Load label mapping
with open('data/roberta_data_info.json', 'r') as f:
    info = json.load(f)
id2label = {int(k): v for k, v in info['id2label'].items()}
print("Label mapping loaded.")

# Load test tensors
test_dict = torch.load('data/roberta_test.pt')
print("Test samples:", len(test_dict['input_ids']))

In [ ]:
class ABSADataset(Dataset):
    def __init__(self, data_dict):
        self.input_ids = data_dict['input_ids']
        self.attention_mask = data_dict['attention_mask']
        self.aspect_labels = data_dict['aspect_labels']
        self.sentiment_labels = data_dict['sentiment_labels']
        self.has_implicit = data_dict['has_implicit']
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.__dict__.items()}

test_dataset = ABSADataset(test_dict)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
print("Test DataLoader ready.")

In [ ]:
model = MultiTaskRoBERTa().to(device)

# Try to load the best model (if exists)
model_path = 'models/multitask_roberta_best.pt'
if not os.path.exists(model_path):
    # Fallback to any seed model
    import glob
    model_files = glob.glob('models/multitask_roberta_seed*.pt')
    if model_files:
        model_path = model_files[0]
        print(f"Using fallback model: {model_path}")
    else:
        raise FileNotFoundError("No trained model found. Please run training first.")

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("Model loaded.")

In [ ]:
# Helper to extract source from XML files
def get_sources(xml_path, source_name):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    sources = []
    for sentence in root.findall('sentence'):
        text_elem = sentence.find('text')
        if text_elem is not None and text_elem.text:
            sources.append(source_name)
    return sources

# Get sources for all sentences (restaurant + laptop)
rest_sources = get_sources('data/SCAPT-ABSA/data/restaurants/Restaurants_Train_v2_Implicit_Labeled.xml', 'rest')
laptop_sources = get_sources('data/SCAPT-ABSA/data/laptops/Laptops_Train_v2_Implicit_Labeled.xml', 'laptop')
all_sources = rest_sources + laptop_sources
print("Total raw sentences:", len(all_sources))

# Use the same split as preprocessing (seed 42, 80/10/10)
random.seed(42)
indices = list(range(len(all_sources)))
random.shuffle(indices)
n_total = len(all_sources)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)
test_indices = indices[n_train + n_val:]

test_sources = [all_sources[i] for i in test_indices]
print("Test set size:", len(test_sources))
print("Restaurant samples:", test_sources.count('rest'))
print("Laptop samples:", test_sources.count('laptop'))

In [ ]:
# Replicate the filtering logic from src.dataset to get sources only for aspect-annotated sentences
def parse_with_source(xml_path, source_name):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    sources = []
    for sentence in root.findall('sentence'):
        text_elem = sentence.find('text')
        if text_elem is None or text_elem.text is None:
            continue
        aspects_elem = sentence.find('aspectTerms')
        if aspects_elem is None:
            continue
        aspects = aspects_elem.findall('aspectTerm')
        if not aspects:
            continue
        sources.append(source_name)
    return sources

# Build sources list for all aspect-annotated sentences (same as all_data in preprocessing)
rest_sources = parse_with_source('data/SCAPT-ABSA/data/restaurants/Restaurants_Train_v2_Implicit_Labeled.xml', 'rest')
laptop_sources = parse_with_source('data/SCAPT-ABSA/data/laptops/Laptops_Train_v2_Implicit_Labeled.xml', 'laptop')
all_sources = rest_sources + laptop_sources
print("Total aspect-annotated sentences:", len(all_sources))  # Should be 3509

# Apply the same split (seed 42, 80/10/10)
random.seed(42)
indices = list(range(len(all_sources)))
random.shuffle(indices)
n_total = len(all_sources)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)
test_indices = indices[n_train + n_val:]

test_sources = [all_sources[i] for i in test_indices]
print("Test set size:", len(test_sources))
print("Restaurant samples in test:", test_sources.count('rest'))
print("Laptop samples in test:", test_sources.count('laptop'))

In [ ]:
all_aspect_preds = []
all_aspect_labels = []
all_sent_preds = []
all_sent_labels = []

model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference"):
        input_ids = batch['input_ids'].to(device)
        att_mask = batch['attention_mask'].to(device)
        aspect_labels = batch['aspect_labels'].to(device)
        sent_labels = batch['sentiment_labels'].to(device)

        aspect_logits, sent_logits = model(input_ids, att_mask)

        # Aspect extraction
        preds = torch.argmax(aspect_logits, dim=-1)
        for i in range(preds.size(0)):
            pred_seq = preds[i].cpu().numpy()
            label_seq = aspect_labels[i].cpu().numpy()
            mask = label_seq != 0
            if mask.any():
                all_aspect_preds.append([id2label[p] for p in pred_seq[mask]])
                all_aspect_labels.append([id2label[l] for l in label_seq[mask]])

        # Sentiment
        sent_preds = torch.argmax(sent_logits, dim=-1)
        all_sent_preds.extend(sent_preds.cpu().numpy())
        all_sent_labels.extend(sent_labels.cpu().numpy())

print(f"Collected {len(all_sent_preds)} sentiment predictions.")
print(f"Collected {len(all_aspect_preds)} aspect predictions.")
assert len(all_sent_preds) == len(test_sources), "Mismatch: predictions vs sources"
assert len(all_aspect_preds) == len(test_sources), "Mismatch: aspect predictions vs sources"

In [ ]:
# Create boolean masks
rest_mask = [src == 'rest' for src in test_sources]
laptop_mask = [src == 'laptop' for src in test_sources]

# Filter aspect predictions
rest_aspect_preds = [p for i, p in enumerate(all_aspect_preds) if rest_mask[i]]
rest_aspect_labels = [l for i, l in enumerate(all_aspect_labels) if rest_mask[i]]
laptop_aspect_preds = [p for i, p in enumerate(all_aspect_preds) if laptop_mask[i]]
laptop_aspect_labels = [l for i, l in enumerate(all_aspect_labels) if laptop_mask[i]]

# Filter sentiment predictions
rest_sent_preds = [p for i, p in enumerate(all_sent_preds) if rest_mask[i]]
rest_sent_labels = [l for i, l in enumerate(all_sent_labels) if rest_mask[i]]
laptop_sent_preds = [p for i, p in enumerate(all_sent_preds) if laptop_mask[i]]
laptop_sent_labels = [l for i, l in enumerate(all_sent_labels) if laptop_mask[i]]

print(f"Restaurant samples: {len(rest_sent_preds)}")
print(f"Laptop samples: {len(laptop_sent_preds)}")

# Compute metrics
rest_aspect_f1 = f1_score(rest_aspect_labels, rest_aspect_preds) if rest_aspect_labels else 0.0
laptop_aspect_f1 = f1_score(laptop_aspect_labels, laptop_aspect_preds) if laptop_aspect_labels else 0.0
rest_sent_acc = accuracy_score(rest_sent_labels, rest_sent_preds) if rest_sent_labels else 0.0
laptop_sent_acc = accuracy_score(laptop_sent_labels, laptop_sent_preds) if laptop_sent_labels else 0.0

In [ ]:
print("\n===== Cross‑Domain Evaluation =====")
print(f"Restaurant test set (trained on mixed domains):")
print(f"  Aspect F1: {rest_aspect_f1:.4f}")
print(f"  Sentiment Accuracy: {rest_sent_acc:.4f}")
print(f"\nLaptop test set (trained on mixed domains):")
print(f"  Aspect F1: {laptop_aspect_f1:.4f}")
print(f"  Sentiment Accuracy: {laptop_sent_acc:.4f}")

In [ ]:
os.makedirs('results', exist_ok=True)
df = pd.DataFrame([
    {'domain': 'restaurant', 'aspect_f1': rest_aspect_f1, 'sentiment_acc': rest_sent_acc},
    {'domain': 'laptop', 'aspect_f1': laptop_aspect_f1, 'sentiment_acc': laptop_sent_acc}
])
df.to_csv('results/cross_domain_evaluation.csv', index=False)
print("Results saved to results/cross_domain_evaluation.csv")